# Lab 4 - Session 2
## Raspberry Pi 5 Deployment, Image/Video Validation, and Live Inference

**Duration:** 2 hours  
**Session 2 marks:** 40 points  
**Official deliverable:** `SecXX_GroupXX_Lab4_Session2.pdf`

Students will synchronize their group fork with the updated instructor
repository, create a CPU-only Python environment without modifying system kernel
packages, test the USB webcam, and deploy pretrained and custom YOLO26n models.

This revision adds:

- a CPU-only verification that correctly allows `nvidia-ml-py`;
- display-safe OpenCV windows that fit a small Raspberry Pi screen;
- validation on static images and recorded videos;
- `--no-display` for tests that only need saved outputs and summaries.


In [ ]:
#@title Student, group, and fork information
GROUP_MEMBER_NAMES = "" #@param {type:"string"}
SECTION_NUMBER = None #@param {type:"integer"}
GROUP_NUMBER = None #@param {type:"integer"}
FORK_OWNER = "" #@param {type:"string"}
FORK_REPOSITORY_NAME = "lab4-megalopa" #@param {type:"string"}
SESSION1_COMMIT_ID = "" #@param {type:"string"}
# PI_KIT_NUMBER = "" #@param {type:"string"}

assert GROUP_MEMBER_NAMES.strip(), "Enter all group-member names."
assert SECTION_NUMBER in (1, 2), "SECTION_NUMBER must be 1 or 2."
assert 1 <= GROUP_NUMBER <= 25, "GROUP_NUMBER must be between 1 and 25."
assert FORK_OWNER.strip(), "Enter the GitHub username that owns the group fork."
assert FORK_REPOSITORY_NAME.strip(), "Enter the exact fork repository name."
assert SESSION1_COMMIT_ID.strip(), "Enter the final Session 1 commit ID."
# assert PI_KIT_NUMBER.strip(), "Enter the Raspberry Pi kit number."

FORK_URL = f"https://github.com/{FORK_OWNER}/{FORK_REPOSITORY_NAME}.git"

print("Group members:", GROUP_MEMBER_NAMES)
print(f"Section {SECTION_NUMBER:02d}, Group {GROUP_NUMBER:02d}")
print("Group fork:", FORK_URL)
print("Session 1 commit:", SESSION1_COMMIT_ID)
# print("Pi kit:", PI_KIT_NUMBER)


## Session 2 workflow

```text
Synchronize group fork with instructor main repository
        ↓
Clone the updated group fork on Raspberry Pi 5
        ↓
Verify commit, baseline model, custom models, images, and videos
        ↓
Create a clean .venv without running apt
        ↓
Install CPU-only PyTorch, Torchvision, and Ultralytics
        ↓
Verify CPU-only PyTorch
        ↓
Scan and test USB webcam indexes
        ↓
Run pretrained baseline
        ↓
Validate custom model on static images
        ↓
Validate custom model on recorded videos
        ↓
Run custom model with live webcam
        ↓
Install ONNX Runtime and compare formats
        ↓
Test confidence and record temperature
        ↓
Upload evidence and export the PDF
```

The main repository must already contain the updated notebook, deployment
script, baseline model, validation images, and at least one validation video.


## Evidence upload helper

Run the next cell once. Each later evidence cell lets you upload one or more screenshots or photographs and displays them directly in the notebook.


In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import Image as DisplayImage, display

EVIDENCE_DIR = Path("/content/session2_evidence")
EVIDENCE_DIR.mkdir(exist_ok=True)

def upload_evidence(label):
    print("Upload evidence for:", label)
    uploaded = files.upload()
    saved = []
    for name, data in uploaded.items():
        path = EVIDENCE_DIR / f"{label}_{Path(name).name}"
        path.write_bytes(data)
        saved.append(path)
        try:
            display(DisplayImage(filename=str(path)))
        except Exception:
            print("Saved:", path)
    return saved


## 1. Verify the prepared Raspberry Pi 5 kit

Run on the Raspberry Pi terminal:

```bash
hostname
cat /proc/device-tree/model; echo
uname -m
uname -r
python3 --version
vcgencmd measure_temp
```

Do not run a kernel upgrade during the laboratory session.


In [ ]:
upload_evidence('01_pi_system')


| Item | Result |
|---|---|
| Raspberry Pi model | TYPE HERE |
| Architecture | TYPE HERE |
| System Python version | TYPE HERE |
| Initial CPU temperature | TYPE HERE |

**Q1.** What output confirms that the device is the expected Raspberry Pi 5 environment? **[2 marks]**  
**Answer:** TYPE HERE

**Q2.** Why should the system Python version and initial CPU temperature be recorded before inference? **[2 marks]**  
**Answer:** TYPE HERE


## 2. Synchronize the fork, clone it, and verify Session 1 files

### 2.1 Synchronize the group fork

Before using the Raspberry Pi, open the group fork on GitHub and select:

```text
Sync fork → Update branch
```

This brings the latest instructor changes into the group fork while preserving
the group's files under `student_work/`.

### 2.2 Clone the synchronized group fork

Replace `FORK_OWNER` and `FORK_REPOSITORY_NAME`:

```bash
cd ~
rm -rf lab4-megalopa-student

git clone \
  https://github.com/anawatbe-spec/lab4-megalopa.git \
  lab4-megalopa-student

cd ~/lab4-megalopa-student

git remote -v
git rev-parse HEAD

ls -lh shared/models/yolo26n.pt
ls -lh shared/pi5/megalopa_detection_usb.py

find student_work/models -maxdepth 2 -type f | sort
find shared/validation/images -maxdepth 1 -type f | sort
find shared/validation/videos -maxdepth 1 -type f | sort
```

Do not run the deployment script yet. Its Python packages are installed in
Section 3.

At least one supported validation video must exist under:

```text
shared/validation/videos/
```


In [ ]:
upload_evidence('02_clone_fork')


| Git item | Result |
|---|---|
| Fork URL | TYPE HERE |
| Commit ID on Pi | TYPE HERE |
| Session 1 commit matches | YES / NO |
| Baseline `yolo26n.pt` found | YES / NO |
| Custom `.pt` model found | YES / NO |
| Custom ONNX model found | YES / NO |

**Q3.** Why does the Pi clone the group fork instead of the instructor upstream repository? **[2 marks]**  
**Answer:** TYPE HERE

**Q4.** Why must the commit ID on the Pi match the final Session 1 commit ID? **[2 marks]**  
**Answer:** TYPE HERE


## 3. Create and verify a CPU-only Python environment

### 3.1 Do not run `apt` during the student session

The Raspberry Pi kits must be prepared by the instructor. Student installation
must not run:

```bash
sudo apt update
sudo apt install ...
```

On a prepared Pi, those commands can trigger unrelated kernel/DKMS
reconfiguration, including old Wi-Fi drivers.

Check that virtual environments are available:

```bash
python3 -m venv --help >/dev/null && echo "venv support: READY"
```

If this fails, stop and ask the instructor to repair the prepared Pi.

### 3.2 Create a clean project environment

```bash
cd ~/lab4-megalopa-student

deactivate 2>/dev/null || true
rm -rf .venv

python3 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
```

### 3.3 Install CPU-only PyTorch first

```bash
python -m pip install \
  torch \
  torchvision \
  --index-url https://download.pytorch.org/whl/cpu
```

### 3.4 Install Ultralytics

```bash
python -m pip install ultralytics
```

Ultralytics installs required dependencies such as OpenCV, NumPy, Pillow,
PyYAML, and `nvidia-ml-py`. The small `nvidia-ml-py` monitoring interface is
allowed; it is not CUDA and does not make the Pi use an NVIDIA GPU.

ONNX Runtime is installed later only when the ONNX comparison begins.

### 3.5 Verify the environment correctly

```bash
python - <<'PY'
import cv2
import torch
import torchvision
import ultralytics

print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    raise RuntimeError("CUDA should not be active on Raspberry Pi 5.")

if torch.version.cuda is not None:
    raise RuntimeError(
        "This PyTorch build contains CUDA support. "
        "Install torch and torchvision from the CPU-only index."
    )

print("CPU-only environment: PASSED")
PY
```

Expected key output:

```text
CUDA available: False
PyTorch CUDA version: None
CPU-only environment: PASSED
```

Do not reject the environment merely because `nvidia-ml-py` is installed.

### 3.6 Verify the updated deployment script

```bash
python shared/pi5/megalopa_detection_usb.py --help
```

The help output must include:

```text
--mode {image,video,camera}
--video-index
--display-width
--display-height
--no-display
--scan-cameras
```

### 3.7 Reactivate the environment in every new terminal

```bash
cd ~/lab4-megalopa-student
source .venv/bin/activate
```


In [ ]:
upload_evidence('03_python_environment')


| Environment item | Result |
|---|---|
| Virtual environment created | YES / NO |
| OpenCV version | TYPE HERE |
| PyTorch version | TYPE HERE |
| Torchvision version | TYPE HERE |
| Ultralytics version | TYPE HERE |
| CUDA available | TRUE / FALSE |
| PyTorch CUDA version | TYPE HERE |
| CPU-only check | PASSED / FAILED |

**Q5.** Why is CPU-only PyTorch installed before Ultralytics on Raspberry Pi 5? **[2 marks]**  
**Answer:** TYPE HERE

**Q6.** Why can `nvidia-ml-py` be present while the environment is still CPU-only? **[2 marks]**  
**Answer:** TYPE HERE


## 4. Detect and test the USB webcam

Keep `(.venv)` active.

### 4.1 Scan all likely camera indexes

```bash
python shared/pi5/megalopa_detection_usb.py --scan-cameras
```

The scanner tests integer indexes and detected `/dev/video*` nodes. Record an
index that reports:

```text
Camera index N: WORKING
```

Set and export the working value for later commands:

```bash
export CAMERA_INDEX=N
echo "$CAMERA_INDEX"
```

Replace `N` with the working integer.

### 4.2 Capture one raw frame before loading a model
```bash
ls /dev/video*
echo "$CAMERA_INDEX"
```
```bash
python - <<'PY'
import os, glob, cv2
from pathlib import Path

i = int(os.environ["CAMERA_INDEX"])
out = Path("student_work/results/pi5/webcam_environment_test.jpg")
out.parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(i, cv2.CAP_V4L2)
for _ in range(10):
    ok, frame = cap.read()
cap.release()

print("Detected devices:", ", ".join(glob.glob("/dev/video*")))
print("Selected camera index:", i)
print("Test frame obtained:", "YES" if ok and frame is not None else "NO")

if not ok or frame is None:
    raise RuntimeError("No frame obtained")

cv2.imwrite(str(out), frame)
cv2.namedWindow("Webcam Test", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Webcam Test", 760, 420)
cv2.moveWindow("Webcam Test", 10, 10)
cv2.imshow("Webcam Test", frame)

print("Check the preview manually:")
print("- Frame in focus: YES / NO")
print("- Reflection level: LOW / MEDIUM / HIGH")

while cv2.getWindowProperty("Webcam Test", cv2.WND_PROP_VISIBLE) >= 1:
    if cv2.waitKey(50) & 0xFF in (ord("q"), ord("Q"), 27, 13, 32):
        break

cv2.destroyAllWindows()
PY
```


In [ ]:
upload_evidence('04_usb_webcam')


| Webcam item | Result |
|---|---|
| Detected `/dev/video*` devices | TYPE HERE |
| Selected camera index | TYPE HERE |
| Test frame obtained | YES / NO |
| Frame in focus | YES / NO |
| Reflection level | LOW / MEDIUM / HIGH |

**Q7.** What does `/dev/video0` or another `/dev/video*` entry represent? **[2 marks]**  
**Answer:** TYPE HERE

**Q8.** Why must a raw webcam frame be tested before loading an AI model? **[2 marks]**  
**Answer:** TYPE HERE


## 5. Run the pretrained YOLO26n baseline

Use the working webcam index found in Section 4:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --baseline \
  --mode camera \
  --camera-index "$CAMERA_INDEX" \
  --confidence 0.25 \
  --imgsz 640 \
  --display-width 760 \
  --display-height 420
```

The preview is automatically scaled to fit a small Pi display. The saved frame
retains its original annotated resolution.

Controls:

- `S`: save an annotated frame;
- `Q` or `Esc`: stop and save the performance summary.


In [ ]:
upload_evidence('05_pretrained_yolo26n_baseline')


| Baseline observation | Result |
|---|---|
| Objects shown to the webcam | TYPE HERE |
| Correctly detected general classes | TYPE HERE |
| Object missed by the model | TYPE HERE |
| False detection observed | TYPE HERE |
| Approximate FPS | TYPE HERE |
| Did it correctly detect the laboratory crab classes? | YES / NO |

**Q9.** What types of objects can the pretrained YOLO26n model detect, and why does it not automatically recognize the laboratory's custom crab classes? **[2 marks]**  
**Answer:** TYPE HERE

**Q10.** Explain how fine-tuning changes a general pretrained detector into a task-specific megalopa or crablet detector. **[2 marks]**  
**Answer:** TYPE HERE


## 6. Validate the custom model on static images

The script lists files in `shared/validation/images/` alphabetically.

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode image \
  --image-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --display-width 760 \
  --display-height 420
```

The revised preview:

- uses a movable `WINDOW_NORMAL` window;
- is positioned near the top-left corner;
- is scaled down to fit within `760 × 420`;
- continues calling `cv2.waitKey()` so the window remains responsive;
- saves the full-resolution annotated image separately.

Close the static preview using `Q`, `Esc`, `Enter`, or `Space`.

When a graphical preview is not needed:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode image \
  --image-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --no-display
```

Outputs are saved under `student_work/results/pi5/custom/`.


In [ ]:
upload_evidence('06_static_inference')


| Static inference item | Result |
|---|---|
| Image index and filename | TYPE HERE |
| Selected model | TYPE HERE |
| Model format | TYPE HERE |
| Actual count | TYPE HERE |
| Predicted count | TYPE HERE |
| Confidence threshold | TYPE HERE |
| Inference latency | TYPE HERE |

**Q11.** Why should static-image inference be completed before live webcam inference? **[2 marks]**  
**Answer:** TYPE HERE

**Q12.** What is the purpose of the confidence threshold? **[2 marks]**  
**Answer:** TYPE HERE

**Q13.** Calculate the absolute count error and state whether the model overcounted or undercounted. **[2 marks]**  
**Answer:** TYPE HERE


## 7. Validate the custom model on a recorded video

Confirm that the synchronized main repository contains at least one supported
file under:

```text
shared/validation/videos/
```

Supported formats include `.mp4`, `.avi`, `.mov`, `.mkv`, `.m4v`, and `.webm`.

List the available videos:

```bash
find shared/validation/videos -maxdepth 1 -type f | sort
```

Run the first indexed validation video:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode video \
  --video-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --display-width 760 \
  --display-height 420
```

The script saves:

```text
student_work/results/pi5/custom/video_<name>_annotated.mp4
student_work/results/pi5/custom/video_<name>_summary.json
student_work/results/pi5/custom/video_<name>_frame_metrics.csv
```

The JSON summary records source FPS, frames processed, average inference
latency, inference FPS, end-to-end processing FPS, and count statistics. The CSV
stores frame index, video timestamp, total detections, latency, rolling FPS, and
class counts for every processed frame.

Controls:

- `S`: save the current annotated frame;
- `Q` or `Esc`: stop the video early.

For a complete non-graphical run:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode video \
  --video-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --no-display
```


In [ ]:
upload_evidence('07_video_validation')

| Video-validation item | Result |
|---|---|
| Validation-video filename | TYPE HERE |
| Source FPS | TYPE HERE |
| Frames processed | TYPE HERE |
| Average inference latency | TYPE HERE |
| Inference FPS | TYPE HERE |
| End-to-end processing FPS | TYPE HERE |
| Mean detections per frame | TYPE HERE |
| Minimum–maximum detections | TYPE HERE |
| Output video path | TYPE HERE |
| Frame-metrics CSV path | TYPE HERE |

**Q14.** Why is validation on a recorded video more demanding than validation on one static image? **[2 marks]**  
**Answer:** TYPE HERE

**Q15.** Compare source FPS, inference FPS, and end-to-end processing FPS. Was the Pi processing the video in real time? **[2 marks]**  
**Answer:** TYPE HERE


## 8. Run the custom model with the live USB webcam

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode camera \
  --camera-index "$CAMERA_INDEX" \
  --confidence 0.25 \
  --imgsz 640 \
  --display-width 760 \
  --display-height 420
```

When `--model` is omitted, select the group model from the numbered list.

Controls:

- `S`: save an annotated frame;
- `Q` or `Esc`: stop and save the live-performance summary.


In [ ]:
upload_evidence('08_live_webcam')

| Live inference item | Result |
|---|---|
| Selected custom model | TYPE HERE |
| Camera index | TYPE HERE |
| Average latency | TYPE HERE |
| Inference FPS | TYPE HERE |
| End-to-end FPS | TYPE HERE |
| Saved live frame | TYPE HERE |

**Q16.** State one webcam, lighting, motion, or display condition that affected live detection, and explain its effect. **[2 marks]**  
**Answer:** TYPE HERE


## 9. Install ONNX Runtime and compare model formats

ONNX Runtime is needed only for this comparison, so it is installed at this
stage rather than during the initial environment setup.

### 9.1 Install and verify ONNX Runtime

Keep `(.venv)` active:

```bash
python -m pip install onnxruntime

python - <<'PY'
import onnxruntime

print("ONNX Runtime:", onnxruntime.__version__)
print("Available providers:", onnxruntime.get_available_providers())
print("ONNX Runtime check: PASSED")
PY
```

### 9.2 Compare PyTorch and ONNX

Run the same indexed validation image using the `.pt` and ONNX versions of the
group model. Keep the image index, confidence, and input size unchanged.

PyTorch example:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode image \
  --model student_work/models/sec01_group01_megalopa_yolo26n_best.pt \
  --image-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --no-display
```

ONNX example:

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode image \
  --model student_work/models/sec01_group01_megalopa_yolo26n_best.onnx \
  --image-index 1 \
  --confidence 0.25 \
  --imgsz 640 \
  --no-display
```

Find the exact filenames using:

```bash
find student_work/models -maxdepth 2 -type f | sort
```


In [ ]:
upload_evidence('09_model_comparison')

| Item | Result |
|---|---|
| ONNX Runtime version | TYPE HERE |
| Available ONNX providers | TYPE HERE |

| Format | Predicted count | Latency | FPS | File size |
|---|---:|---:|---:|---:|
| `.pt` | TYPE HERE | TYPE HERE | TYPE HERE | TYPE HERE |
| ONNX | TYPE HERE | TYPE HERE | TYPE HERE | TYPE HERE |

**Q17.** Which format was faster, and were the predicted counts equivalent? **[2 marks]**  
**Answer:** TYPE HERE

**Q18.** Why is the fastest model format not automatically the most accurate or most suitable? **[2 marks]**  
**Answer:** TYPE HERE


## 10. Test another confidence threshold and record temperature

Run one additional custom-model test using the same image and input size but a
different confidence threshold. Use `--no-display` to avoid reopening a window:

record the Raspberry Pi temperature before:
```bash
vcgencmd measure_temp
```

```bash
python shared/pi5/megalopa_detection_usb.py \
  --mode image \
  --image-index 1 \
  --confidence 0.50 \
  --imgsz 640 \
  --no-display
```

Then record the Raspberry Pi temperature after:

```bash
vcgencmd measure_temp
```


In [ ]:
upload_evidence('10_confidence_temperature')

| Observation | Result |
|---|---|
| Alternative confidence | TYPE HERE |
| Effect on detections | TYPE HERE |
| Initial temperature | TYPE HERE |
| Final temperature | TYPE HERE |

**Q19.** Explain how confidence affects false positives and missed detections, and why CPU temperature is monitored. **[2 marks]**  
**Answer:** TYPE HERE


## 11. Final evidence and traceability

Capture final evidence showing:

- the Raspberry Pi 5 kit and USB webcam;
- the scaled live custom-model window;
- the selected model name;
- total detections and class counts;
- inference latency and FPS;
- the saved static-image output;
- the saved annotated validation video and its JSON summary.


In [ ]:
upload_evidence('11_final_system')

| Final item | Result |
|---|---|
| Group fork URL | TYPE HERE |
| Synced commit ID | TYPE HERE |
| CPU-only environment passed | YES / NO |
| Working camera index | TYPE HERE |
| Custom model used | TYPE HERE |
| Static-image output | TYPE HERE |
| Annotated-video output | TYPE HERE |
| Video summary JSON | TYPE HERE |
| Live average latency | TYPE HERE |
| Live inference FPS | TYPE HERE |

**Q20.** Explain how the fork URL, synced commit ID, environment verification, model filename, validation media, and saved Pi outputs make the deployment reproducible. **[2 marks]**  
**Answer:** TYPE HERE

## Session 2 PDF submission

1. Confirm all 20 answers are complete.
2. Confirm the CPU-only check displays `PASSED`.
3. Confirm the working webcam index is recorded.
4. Confirm static-image, validation-video, and live-webcam evidence is visible.
5. Confirm the `.pt` versus ONNX comparison is complete.
6. Use **File → Print → Save as PDF**.
7. Submit `SecXX_GroupXX_Lab4_Session2.pdf`.
8. Do not display passwords or access tokens.
